# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
import os

PROJECT_ROOT = r"/root/autodl-tmp/sandbox/5329_ziprun/5329_QANet-zyt-v2"
print("Project root:", PROJECT_ROOT)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install Python dependencies needed by the notebook and Experiment 3
import subprocess, sys, os
req_file = os.path.join(PROJECT_ROOT, "requirements.txt")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", req_file])
for pkg in ["matplotlib", "scipy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
print("Dependencies ready.")

---
## Section 0 — Environment Setup

Set the project root and install dependencies.

In [ ]:
import sys, os
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:
print("Section 3 skipped for this run.")

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
print("Section 4 skipped for this run.")

---
# Section 5 — Stage 3: Mechanism-Oriented Experiments

This run executes only the modified **Experiment 3** after running the earlier setup/download/preprocess sections from scratch.

In [ ]:
# Modified Experiment 3 shared config
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from TrainTools.train import train
from EvaluateTools.evaluate import evaluate

BASELINE_CONFIG = dict(
    batch_size=32,
    num_steps=60000,
    checkpoint=200,
    early_stop=30,
    optimizer_name="adam",
    scheduler_name="lambda",
    learning_rate=1e-3,
    beta1=0.8,
    beta2=0.999,
    eps=1e-7,
    weight_decay=3e-6,
    warmup_steps=1000,
    ema_decay=0.9999,
    d_model=128,
    dropout=0.1,
    dropout_char=0.05,
)

EVAL_CONFIG = dict(
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    dev_eval_json="_data/dev_eval.json",
    d_model=128,
)

SEEDS = [42, 13]
print("Modified Experiment 3 config ready.")


---
## Experiment 1 — Skipped

This run skips Experiment 1 and keeps only the modified Experiment 3.


In [ ]:
print("Experiment 1 skipped for this run.")


In [ ]:
print("Experiment 1 evaluation skipped for this run.")


In [ ]:
print("Experiment 1 plotting skipped for this run.")


In [ ]:
print("Experiment 1 diagnostics skipped for this run.")


---
## Experiment 2 — Skipped

This run skips Experiment 2 and keeps only the modified Experiment 3.


In [ ]:
print("Experiment 2 skipped for this run.")


In [ ]:
print("Experiment 2 evaluation skipped for this run.")


In [ ]:
print("Experiment 2 plotting skipped for this run.")


In [ ]:
print("Experiment 2 diagnostics skipped for this run.")


---
## Experiment 3 — Is Normalization Itself Necessary?

Control: `layer_norm`
Ablation: `identity`

This modified version runs only Experiment 3 after the earlier setup/download/preprocess stages.

In [ ]:
# ── Experiment 3: Register IdentityNorm ─────────────────────────────────────
import torch
import torch.nn as nn
from Models.Normalizations import normalizations
from Models.Normalizations import normalization as _norm_module

class IdentityNorm(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
    def forward(self, x):
        return x

normalizations["identity"] = IdentityNorm
_orig_get_norm = _norm_module.get_norm

def _patched_get_norm(name, d_model, length, num_groups=8):
    if name == "identity":
        return IdentityNorm()
    return _orig_get_norm(name, d_model, length, num_groups)

_norm_module.get_norm = _patched_get_norm
print("IdentityNorm ready.")

In [ ]:
# Experiment 3: Train selected groups
import os
import json
import time

exp3_groups = [
    ("A_layer_norm", "layer_norm"),
    ("B_identity", "identity"),
]

exp3_results = {}
os.makedirs(os.path.join("_exp", "exp3"), exist_ok=True)

for group_name, norm in exp3_groups:
    for seed in SEEDS:
        run_tag = f"exp3_{group_name}_seed{seed}"
        save_dir = os.path.join("_exp", "exp3", run_tag)
        log_dir = os.path.join(save_dir, "log")
        print(f"\n{'=' * 60}")
        print(f"Experiment 3 | {group_name} | seed={seed} | norm_name={norm}")
        print(f"{'=' * 60}\n")
        start_time = time.time()
        results = train(**BASELINE_CONFIG, seed=seed, save_dir=save_dir, log_dir=log_dir, norm_name=norm)
        elapsed = time.time() - start_time
        exp3_results[run_tag] = {
            "group": group_name,
            "norm": norm,
            "seed": seed,
            "best_f1": results["best_f1"],
            "best_em": results["best_em"],
            "history": results["history"],
            "elapsed_sec": elapsed,
        }
        with open(os.path.join("_exp", "exp3", "results.json"), "w") as f:
            json.dump({k: {kk: vv for kk, vv in v.items() if kk != "history"} for k, v in exp3_results.items()}, f, indent=2)

print("Experiment 3 training complete.")


In [ ]:
# ── Experiment 3: Full Dev Evaluation ────────────────────────────────────────
import os
import numpy as np

exp3_eval = {}
for group_name, norm in exp3_groups:
    f1_list, em_list = [], []
    for seed in SEEDS:
        run_tag = f"exp3_{group_name}_seed{seed}"
        save_dir = os.path.join("_exp", "exp3", run_tag)
        log_dir = os.path.join("_exp", "exp3", run_tag, "log")
        metrics = evaluate(**EVAL_CONFIG, save_dir=save_dir, log_dir=log_dir, norm_name=norm)
        f1_list.append(metrics["f1"])
        em_list.append(metrics["exact_match"])
        print(f"  {run_tag}: F1={metrics['f1']:.4f}  EM={metrics['exact_match']:.4f}")
    exp3_eval[group_name] = {
        "f1_mean": np.mean(f1_list), "f1_std": np.std(f1_list),
        "em_mean": np.mean(em_list), "em_std": np.std(em_list),
        "f1_runs": f1_list, "em_runs": em_list,
    }
print("\n" + "=" * 60)
print("Experiment 3 Summary (Full Dev Set)")
print("=" * 60)
for group, vals in exp3_eval.items():
    print(f"  {group:20s}:  F1 = {vals['f1_mean']:.4f} ± {vals['f1_std']:.4f}  |  EM = {vals['em_mean']:.4f} ± {vals['em_std']:.4f}")

In [ ]:
# ── Experiment 3: Training Curves ────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"A_layer_norm": "tab:blue", "B_identity": "tab:red"}
for group_name, norm in exp3_groups:
    color = colors[group_name]
    all_dev_f1, all_dev_loss, all_train_loss = [], [], []
    for seed in SEEDS:
        run_tag = f"exp3_{group_name}_seed{seed}"
        hist = exp3_results[run_tag]["history"]
        steps = [h["step"] for h in hist]
        all_dev_f1.append([h["dev_f1"] for h in hist])
        all_dev_loss.append([h["dev_loss"] for h in hist])
        all_train_loss.append([h["train_loss"] for h in hist])
    min_len = min(len(x) for x in all_dev_f1)
    steps = steps[:min_len]
    dev_f1_arr = np.array([x[:min_len] for x in all_dev_f1])
    dev_loss_arr = np.array([x[:min_len] for x in all_dev_loss])
    train_loss_arr = np.array([x[:min_len] for x in all_train_loss])
    axes[0].plot(steps, dev_f1_arr.mean(0), color=color, label=group_name)
    axes[0].fill_between(steps, dev_f1_arr.mean(0)-dev_f1_arr.std(0), dev_f1_arr.mean(0)+dev_f1_arr.std(0), alpha=0.15, color=color)
    axes[1].plot(steps, dev_loss_arr.mean(0), color=color, label=group_name)
    axes[1].fill_between(steps, dev_loss_arr.mean(0)-dev_loss_arr.std(0), dev_loss_arr.mean(0)+dev_loss_arr.std(0), alpha=0.15, color=color)
    axes[2].plot(steps, train_loss_arr.mean(0), color=color, label=group_name)
for ax in axes:
    ax.set_xlabel("Step")
    ax.grid(True, alpha=0.3)
axes[0].set_title("Dev F1 vs Steps"); axes[0].set_ylabel("F1"); axes[0].legend()
axes[1].set_title("Dev Loss vs Steps"); axes[1].set_ylabel("Loss"); axes[1].legend()
axes[2].set_title("Train Loss vs Steps"); axes[2].set_ylabel("Loss"); axes[2].legend()
fig.suptitle("Experiment 3: LayerNorm vs Identity", fontsize=14, y=1.02)
os.makedirs(os.path.join("_exp", "exp3"), exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("_exp", "exp3", "exp3_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Experiment 3: Minimal follow-up analysis ─────────────────────────────────
print("Comparing steps to reach best dev F1 (seed=42) ...\n")
for group_name, norm in exp3_groups:
    run_tag = f"exp3_{group_name}_seed42"
    if run_tag in exp3_results:
        hist = exp3_results[run_tag]["history"]
        best_step = max(hist, key=lambda h: h["dev_f1"])["step"]
        best_f1 = max(h["dev_f1"] for h in hist)
        total_steps = hist[-1]["step"] if hist else 0
        elapsed = exp3_results[run_tag].get("elapsed_sec")
        elapsed_str = f", wall time {elapsed/60:.1f} min" if elapsed else ""
        print(f"  {group_name:<16}: best dev F1 = {best_f1:.4f} at step {best_step} (trained {total_steps} steps total{elapsed_str})")

---
## All Experiments — Summary Table

In [ ]:
# ── Experiment 3: Summary Table ─────────────────────────────────────────────
print("=" * 80)
print("  EXPERIMENT 3 RESULTS — Full Dev Set (mean ± std over 2 seeds)")
print("=" * 80)
print(f"  {'Group':<22} {'F1':>20} {'EM':>20}")
print(f"  {'-' * 62}")
for group, vals in exp3_eval.items():
    print(f"  {group:<22} {vals['f1_mean']:>8.4f} ± {vals['f1_std']:<8.4f} {vals['em_mean']:>8.4f} ± {vals['em_std']:<8.4f}")
print("\n" + "=" * 80)